In [ ]:
import json
import requests
from pathlib import Path
from datetime import date
from calendar import monthrange
from tqdm import tqdm
from pystac_client import Client


In [ ]:

# =============================
# USER SETTINGS
# =============================

STAC_URL = "https://catalogue.dataspace.copernicus.eu/stac"
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

OUTDIR = Path("S2_L1C")
OUTDIR.mkdir(exist_ok=True)

CLOUD_MAX = 10  # percent





In [ ]:

GEOJSON_FILE = "ROI\Bobcoast50km.geojson"

# =============================
# LOAD AOI (GeoJSON → geometry)
# =============================

with open(GEOJSON_FILE) as f:
    gj = json.load(f)

geom = gj["features"][0]["geometry"]  # MultiPolygon OK

print(geom)


In [ ]:

# =============================
# AUTHENTICATION (interactive)
# =============================

print(" Authenticating with Copernicus Dataspace...")

username = input("Copernicus username: ")
password = input("Copernicus password: ")



In [ ]:


token_resp = requests.post(
    TOKEN_URL,
    data={
        "grant_type": "password",
        "client_id": "cdse-public",
        "username": username,
        "password": password,
    },
)

token_resp.raise_for_status()
access_token = token_resp.json()["access_token"]

HEADERS = {"Authorization": f"Bearer {access_token}"}

print("Authentication successful")

# =============================
# CONNECT STAC
# =============================

cat = Client.open(STAC_URL)


In [ ]:

# =============================
# DOWNLOAD FUNCTION
# =============================

def download_safe(item, outdir, headers):
    product_id = item["id"]
    url = f"https://download.dataspace.copernicus.eu/odata/v1/Products('{product_id}')/$value"

    outfile = outdir / f"{product_id}.SAFE.zip"
    if outfile.exists():
        print(f"✔ Already exists: {outfile.name}")
        return

    print(f"⬇ Downloading {product_id}")

    with requests.get(url, headers=headers, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))

        with open(outfile, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=outfile.name
        ) as pbar:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))


In [ ]:

# =============================
# YEAR / MONTH LOOP
# =============================

while True:
    year_input = input("\nEnter year (e.g. 2017) or 'q' to quit: ").strip()
    if year_input.lower() == "q":
        print("Exiting.")
        break

    month_input = input("Enter month (1–12): ").strip()

    try:
        year = int(year_input)
        month = int(month_input)
        if not (1 <= month <= 12):
            raise ValueError
    except ValueError:
        print(" Invalid year or month")
        continue

    start = date(year, month, 1)
    end = date(year, month, monthrange(year, month)[1])

    outdir = OUTDIR / str(year) / f"{month:02d}"
    outdir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Searching {year}-{month:02d} ===")

    params = {
        "collections": "sentinel-2-l1c",
        "datetime": f"{start.isoformat()}/{end.isoformat()}",
        "intersects": geom,
        "query": {
            "eo:cloud_cover": {"gte": 0, "lte": CLOUD_MAX}
        },
    }

    items = list(cat.search(**params).items_as_dicts())

    print(f"Found {len(items)} scenes")

    if len(items) == 0:
        continue

    for item in items:
        download_safe(item, outdir, HEADERS)
